# Unit 8 — Regular Expressions and Threads
**Course:** BT151CO — Object-Oriented Programming with Python  
**Unit:** 8 of 8  
**Duration:** 4 Hours  

---

## What Is This Notebook?

This notebook covers two powerful Python topics in one unit:

- **Part A — Regular Expressions (`re` module):** Find, validate, extract, and replace text using patterns. No installation needed — `re` is part of the standard library.
- **Part B — Threads (`threading` module):** Run multiple tasks concurrently. Also built into the standard library.
- **Part C — Django Overview:** A conceptual look at how everything in this course comes together in a real web framework.

> **All code in Parts A and B is fully runnable inside this notebook.**  
> Part C cells are marked as *read-only examples* — Django requires a project directory to run.

---

## Learning Objectives

By the end of this unit you will be able to:

1. Write regular expressions to **search**, **validate**, and **transform** text
2. Use `re.split()`, `re.match()`, `re.search()`, `re.findall()`, and `re.sub()`
3. Apply quantifiers, character classes, and capturing groups correctly
4. Create and start threads using `threading.Thread`
5. Protect shared data from race conditions using `Lock`
6. Coordinate threads using `Semaphore` and `Event`
7. Describe Django's MVT architecture and how it connects OOP, databases, and networking


## Table of Contents

**Part A — Regular Expressions**

1. [What Is a Regular Expression?](#s1)
2. [re.split() — Splitting on a Pattern](#s2)
3. [Special Characters and Metacharacters](#s3)
4. [Character Classes and Predefined Sequences](#s4)
5. [Working with Dates — Patterns and Groups](#s5)
6. [Matching Email Addresses](#s6)
7. [Quantifiers](#s7)
8. [re.match() — Validation](#s8)
9. [re.findall() — Finding All Matches](#s9)
10. [re.sub() — Find and Replace](#s10)
11. [re.search() vs re.match()](#s11)
12. [re.compile() and Flags](#s12)


---
# Part A — Regular Expressions

<a id='s1'></a>
## Section 1: What Is a Regular Expression?

A **regular expression** (regex) is a pattern written in a special mini-language that describes the *shape* of the text you are looking for.

**Think of a form template:**  
A form has boxes like `[Date] __/__/____`.  
The box shape tells you what goes there — two digits, a slash, two more digits, etc.  
A regex does the same thing in code.

| Task | Without Regex | With Regex |
|---|---|---|
| Extract all emails from 10 000 lines | Hundreds of `if` statements | One line |
| Validate phone number format | Multiple `split()` and checks | One `match()` call |
| Convert date format | Manual string slicing | One `sub()` call |

> **Always import `re` first.** It is part of the Python standard library — no `pip install`.

```python
import re
```

> **Always use raw strings** `r"..."` for patterns.  
> Without `r`, Python processes backslashes before the regex engine sees them.  
> `\s` without `r` becomes a tab character — not the whitespace class!

In [34]:
import re

# One line to extract all UK phone numbers from a long string
text = (
    "Contact Alice on 07700 900123 or Bob on 07911 654321. "
    "Or email charlie@example.com"
)

phones = re.findall(r"\b07\d{3}\s?\d{6}\b", text)
print("Phones found:", phones)

emails = re.findall(r"[\w.]+@[\w.]+\.[a-zA-Z]{2,}", text)
print("Emails found:", emails)

Phones found: ['07700 900123', '07911 654321']
Emails found: ['charlie@example.com']


<a id='s2'></a>
## Section 2: `re.split()` — Splitting on a Pattern

Python's built-in `str.split()` only handles **one separator** at a time.  
`re.split()` handles **any pattern** — including multiple separators at once.

**Pattern breakdown: `[,;\s]+`**

| Part | Meaning |
|---|---|
| `[,;\s]` | Any one of: comma, semicolon, whitespace |
| `+` | One or more of the above |

In [31]:
import re

# Student names separated by a mix of commas, semicolons, and spaces
student_list = "Alice, Bob;  Charlie,   Diana; Eve"

# Built-in split — only handles ONE separator
print("split(','):", student_list.split(","))
# ['Alice', ' Bob;  Charlie', '   Diana; Eve']  <- messy!

split(','): ['Alice', ' Bob;  Charlie', '   Diana; Eve']


In [32]:
# re.split() — handles ALL separators at once
parts = re.split(r"[,;\s]+", student_list)
print("re.split():", parts)
# ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve']  ✓

re.split(): ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve']


In [33]:
# maxsplit — limit the number of splits
entry = "BT2024:Alice:Maths:Grade A"
student_id, rest = re.split(r":", entry, maxsplit=1)
print(f"ID: {student_id}, rest: {rest}")
# ID: BT2024, rest: Alice:Maths:Grade A

ID: BT2024, rest: Alice:Maths:Grade A


<a id='s3'></a>
## Section 3: Special Characters — Metacharacters

These characters have **special meaning** in a regex pattern:

```
. ^ $ * + ? { } [ ] \ | ( )
```

| Symbol | Name | Meaning | Example |
|---|---|---|---|
| `.` | Dot | Any single character (not newline) | `c.t` → cat, cut, c9t |
| `^` | Caret | Start of the string | `^Dear` → starts with 'Dear' |
| `$` | Dollar | End of the string | `\.pdf$` → ends with '.pdf' |
| `\` | Backslash | Escape OR introduce a sequence | `\.` = literal dot |
| `\|` | Pipe | OR — try left, then right | `cat\|dog` = 'cat' or 'dog' |

> **To match a literal metacharacter, escape it with `\`**  
> `£\d+\.\d{2}` matches `£2.50` — the `\.` is a literal dot.

In [28]:
import re

# Dot — matches any single character
words = ["cat", "cut", "cot", "coat", "c9t"]
matches = [w for w in words if re.match(r"c.t", w)]
print("c.t matches:", matches)   # ['cat', 'cut', 'cot', 'c9t']

c.t matches: ['cat', 'cut', 'cot', 'c9t']


In [29]:
# Escaped dot — must be a literal dot
prices = ["£1.99", "£10.00", "£5x00"]
valid  = [p for p in prices if re.match(r"£\d+\.\d{2}", p)]
print("Valid prices:", valid)    # ['£1.99', '£10.00']

Valid prices: ['£1.99', '£10.00']


In [30]:
# Pipe — OR
genres = ["science fiction", "mystery", "fantasy", "horror", "romance"]
popular = [g for g in genres if re.search(r"fiction|fantasy", g)]
print("Popular genres:", popular)  # ['science fiction', 'fantasy']

Popular genres: ['science fiction', 'fantasy']


<a id='s4'></a>
## Section 4: Character Classes and Predefined Sequences

A **character class** `[...]` matches any ONE character from the set.

| Pattern | Matches |
|---|---|
| `[aeiouAEIOU]` | Any vowel |
| `[a-z]` | Any lowercase letter |
| `[A-Z]` | Any uppercase letter |
| `[0-9]` | Any digit |
| `[^aeiou]` | Any character that is NOT a vowel |

**Predefined shorthand sequences:**

| Sequence | Meaning | Equivalent |
|---|---|---|
| `\d` | Any digit | `[0-9]` |
| `\D` | Any non-digit | `[^0-9]` |
| `\w` | Word character (letter, digit, `_`) | `[a-zA-Z0-9_]` |
| `\W` | Non-word character | `[^a-zA-Z0-9_]` |
| `\s` | Whitespace (space, tab, newline) | `[ \t\n\r]` |
| `\S` | Non-whitespace | `[^ \t\n\r]` |
| `\b` | Word boundary (zero-width) | Position between word/non-word |

> **`^` inside `[...]`** means NOT.  
> **`^` outside `[...]`** means start of string. Context matters!

In [25]:
import re

# Character classes — find vowels and consonants
name = "Bartholomew"
vowels = re.findall(r"[aeiouAEIOU]", name)
print(f"Vowels in '{name}': {vowels}")   # ['a', 'o', 'o', 'e', 'o']

Vowels in 'Bartholomew': ['a', 'o', 'o', 'e']


In [26]:
# Predefined \d — extract all numbers from a game log
game_log = "Player1 scored 1200 points. Player2 scored 850 points."
scores = re.findall(r"\b\d+\b", game_log)
print("All numbers:", scores)    # ['1', '1200', '2', '850']

All numbers: ['1200', '850']


In [27]:
# \b word boundary — find whole word 'cat', not 'scatter'
text = "the cat scattered the catalogue"
without_b = re.findall(r"cat", text)
with_b    = re.findall(r"\bcat\b", text)
print("Without \\b:", without_b)  # ['cat', 'cat', 'cat']
print("With \\b:   ", with_b)     # ['cat']

Without \b: ['cat', 'cat', 'cat']
With \b:    ['cat']


<a id='s5'></a>
## Section 5: Working with Dates — Patterns and Capturing Groups

**Match UK date format `DD/MM/YYYY`:**
- `\d{2}` — exactly two digits
- `/` — literal slash
- `\d{4}` — exactly four digits

**Capturing groups `(...)`** let you extract specific parts of a match.  
- `re.findall()` with **one group** returns a list of strings.  
- `re.findall()` with **multiple groups** returns a list of **tuples**.

**Named groups `(?P<name>...)`** make your patterns self-documenting.

In [22]:
import re

borrow_log = (
    'Alice borrowed "Python Cookbook" on 01/03/2024, due back 15/03/2024.\n'
    'Bob borrowed "Clean Code" on 22/02/2024, due back 07/03/2024.'
)

# 1. Find all dates
dates = re.findall(r"\d{2}/\d{2}/\d{4}", borrow_log)
print("All dates:", dates)

All dates: ['01/03/2024', '15/03/2024', '22/02/2024', '07/03/2024']


In [23]:
# 2. Capturing groups — extract day, month, year as tuples
pattern = r"(\d{2})/(\d{2})/(\d{4})"
for day, month, year in re.findall(pattern, borrow_log):
    print(f"  ISO format: {year}-{month}-{day}")

  ISO format: 2024-03-01
  ISO format: 2024-03-15
  ISO format: 2024-02-22
  ISO format: 2024-03-07


In [24]:
# 3. Named groups — even more readable
named = r"(?P<day>\d{2})/(?P<month>\d{2})/(?P<year>\d{4})"
for match in re.finditer(named, borrow_log):
    d = match.group('day')
    m = match.group('month')
    y = match.group('year')
    print(f"  Named group: {y}-{m}-{d}")

  Named group: 2024-03-01
  Named group: 2024-03-15
  Named group: 2024-02-22
  Named group: 2024-03-07


<a id='s6'></a>
## Section 6: Matching Email Addresses

Email addresses have a predictable structure: `local@domain.tld`

**Simple pattern:** `\S+@\S+\.\S+`  
- `\S+` = one or more non-whitespace characters
- `@` = literal at-sign
- `\.` = literal dot (not a metacharacter here)

**More precise:** `[\w.\-]+@[\w.\-]+\.[a-zA-Z]{2,}`  
- `[\w.\-]+` = letters, digits, underscore, dot, hyphen
- `[a-zA-Z]{2,}` = TLD must be at least 2 letters

> Real email validation is very complex. For production code, use a library  
> or just check for `@` and send a confirmation email.

In [19]:
import re

registration_data = """
alice.smith@university.edu
b.jones99@college.ac.uk
charlie_b@student.org
not-an-email
diana@example.com
5555-5555  (phone, not email)
"""

# Simple pattern
simple_emails = re.findall(r"\S+@\S+\.\S+", registration_data)
print("Simple:", simple_emails)

Simple: ['alice.smith@university.edu', 'b.jones99@college.ac.uk', 'charlie_b@student.org', 'diana@example.com']


In [20]:
# More precise pattern
precise_emails = re.findall(
    r"[\w.\-]+@[\w.\-]+\.[a-zA-Z]{2,}",
    registration_data
)
print("Precise:", precise_emails)

Precise: ['alice.smith@university.edu', 'b.jones99@college.ac.uk', 'charlie_b@student.org', 'diana@example.com']


In [21]:
# Validate one email address
def is_valid_email(email):
    pattern = r"^[\w.\-]+@[\w.\-]+\.[a-zA-Z]{2,}$"
    return re.match(pattern, email) is not None

test_emails = ["alice@uni.edu", "not-an-email", "bob@", "@domain.com"]
for e in test_emails:
    print(f"  {e!r:30s} valid={is_valid_email(e)}")

  'alice@uni.edu'                valid=True
  'not-an-email'                 valid=False
  'bob@'                         valid=False
  '@domain.com'                  valid=False


<a id='s7'></a>
## Section 7: Quantifiers — How Many Times?

| Quantifier | Meaning | Example | Matches |
|---|---|---|---|
| `*` | 0 or more | `go*gle` | `ggle`, `gogle`, `google` |
| `+` | 1 or more | `go+gle` | `gogle`, `google` (NOT `ggle`) |
| `?` | 0 or 1 (optional) | `colou?r` | `color`, `colour` |
| `{n}` | Exactly n | `\d{4}` | `2024` |
| `{n,m}` | Between n and m | `\d{2,4}` | `12`, `123`, `1234` |
| `{n,}` | n or more | `\d{3,}` | `100`, `9999`, `123456` |

**Greedy vs Non-Greedy:**  
By default, quantifiers are **greedy** — they match as **much** as possible.  
Add `?` after any quantifier to make it **non-greedy** (match as **little** as possible).

| Pattern | Type | Behaviour |
|---|---|---|
| `.*` | Greedy | Matches as many characters as possible |
| `.*?` | Non-greedy | Matches as few characters as possible |

In [16]:
import re

# Student IDs: exactly 2 uppercase letters + 4 digits
test_ids = ["BT2024", "CS1999", "B2024", "BTS12345", "ab9999", "CS0000"]
pattern  = re.compile(r"^[A-Z]{2}\d{4}$")
for sid in test_ids:
    status = "VALID" if pattern.match(sid) else "INVALID"
    print(f"  {sid:12s} → {status}")

print()

  BT2024       → VALID
  CS1999       → VALID
  B2024        → INVALID
  BTS12345     → INVALID
  ab9999       → INVALID
  CS0000       → VALID



In [17]:
# Greedy vs non-greedy on HTML-like data
html = "<item>Milk</item><item>Bread</item>"

greedy  = re.findall(r"<.*>",  html)
lazy    = re.findall(r"<.*?>", html)
print("Greedy  :", greedy)
print("Non-greedy:", lazy)

Greedy  : ['<item>Milk</item><item>Bread</item>']
Non-greedy: ['<item>', '</item>', '<item>', '</item>']


In [18]:
# Optional character: colour / color
words = ["colour", "color", "colouur", "Colors"]
for w in words:
    m = re.match(r"colou?r", w, re.IGNORECASE)
    print(f"  {w!r:12s} match={bool(m)}")

  'colour'     match=True
  'color'      match=True
  'colouur'    match=False
  'Colors'     match=True


<a id='s8'></a>
## Section 8: `re.match()` — Validation

`re.match()` checks the pattern **only at the beginning** of the string.

Use `^` (start) and `$` (end) anchors to force the **entire string** to match a pattern.

**Match object methods:**

| Method | Returns |
|---|---|
| `.group()` | The entire matched text |
| `.group(1)` | Content of capturing group 1 |
| `.group('name')` | Content of named group |
| `.start()` | Start position of the match |
| `.end()` | End position of the match |
| `.span()` | `(start, end)` tuple |

In [2]:
import re

# Validate student ID: 2 uppercase letters + 4 digits, nothing else
def validate_student_id(sid):
    pattern = r"^[A-Z]{2}\d{4}$"  # ^ = start, $ = end
    match = re.match(pattern, sid)
    if match:
        print(f"  VALID:   '{sid}'  matched: '{match.group()}'")
    else:
        print(f"  INVALID: '{sid}'")

test_cases = ["BT2024", "CS1999", "cs9999", "BTS12345", "CS2025", "12ABCD"]
for t in test_cases:
    validate_student_id(t)

print()

  VALID:   'BT2024'  matched: 'BT2024'
  VALID:   'CS1999'  matched: 'CS1999'
  INVALID: 'cs9999'
  INVALID: 'BTS12345'
  VALID:   'CS2025'  matched: 'CS2025'
  INVALID: '12ABCD'



Syntax for Named Group:

```
(?P<name>pattern)
```

For example: 
```
(?P<author>\w+)
│  │      │
│  │      └── actual regex pattern
│  └───────── name of the group
└──────────── Python syntax for a named group
```

In [ ]:
# Named groups in match
entry = "Lutz_LearningPython_2013"
m = re.match(r"^(?P<author>\w+)_(?P<title>\w+)_(?P<year>\d{4})$", entry)
if m:
    print(f"Author: {m.group('author')}")
    print(f"Title : {m.group('title')}")
    print(f"Year  : {m.group('year')}")
    print(f"Span  : {m.span()}")

> Here, in the regex `^(?P<author>\w+)_(?P<title>\w+)_(?P<year>\d{4})$`, what is `P<author>`? Is it also a regex? 
>
> TODO

<a id='s9'></a>
## Section 9: `re.findall()` — Finding All Matches

`re.findall()` searches the **entire string** and returns a **list** of every match.

**Return type depends on groups:**

| Pattern | `findall()` returns |
|---|---|
| No groups | List of strings |
| One group `(...)` | List of strings (group content only) |
| Multiple groups | List of tuples |

Use `re.finditer()` when you also need the **position** of each match  
(it returns an iterator of Match objects).

In [ ]:
import re

game_report = """
Round 1: Alice scored 1200, Bob scored 850.
Round 2: Alice scored 980,  Bob scored 1650.
Round 3: Alice scored 2100, Bob scored 1100.
"""

# No groups — list of strings
all_numbers = re.findall(r"\d+", game_report)
print("All numbers:", all_numbers)

# \b word boundary — whole numbers only, 4+ digits (1000+)
high_scores = re.findall(r"\b\d{4,}\b", game_report)
print("High scores (1000+):", high_scores)

# One group — list of strings (the group content)
prices_text = "£1.09, £4.99, £12.50, £0.79"
amounts = re.findall(r"£(\d+\.\d{2})", prices_text)
print("Amounts:", amounts)
print(f"Total: £{sum(float(a) for a in amounts):.2f}")

# Multiple groups — list of tuples
log = "Alice: 1200, Bob: 850, Charlie: 1650"
pairs = re.findall(r"([A-Za-z]+):\s*(\d+)", log)
print("Name-score pairs:", pairs)
for name, score in pairs:
    print(f"  {name}: {score}")

<a id='s10'></a>
## Section 10: `re.sub()` — Find and Replace

`re.sub(pattern, replacement, string)` replaces **every match** and returns a **new string**.

> ⚠️ **IMPORTANT:** `sub()` does NOT modify the original string.  
> Always assign the result: `text = re.sub(..., text)`

**Back-references in the replacement string:**  
`\1` refers to capturing group 1, `\2` to group 2, etc.

**Function as replacement:**  
You can pass a function instead of a string. The function receives each Match object  
and returns the replacement string.

In [13]:
import re

# 1. Back-references — convert UK dates to ISO format
records = "Alice: 15/03/2024, Bob: 22/02/2024, Charlie: 01/01/2023"

iso_records = re.sub(
    r"(\d{2})/(\d{2})/(\d{4})",
    r"\3-\2-\1",      # year-month-day
    records
)
print("Original:", records)
print("ISO     :", iso_records)

print()

Original: Alice: 15/03/2024, Bob: 22/02/2024, Charlie: 01/01/2023
ISO     : Alice: 2024-03-15, Bob: 2024-02-22, Charlie: 2023-01-01



In [14]:
# 2. Function as replacement — double every score
def double_score(match):
    original = int(match.group())
    return str(original * 2)

scores = "Alice: 500, Bob: 800, Charlie: 1200"
boosted = re.sub(r"\d+", double_score, scores)
print("Original:", scores)
print("Boosted :", boosted)

print()

Original: Alice: 500, Bob: 800, Charlie: 1200
Boosted : Alice: 1000, Bob: 1600, Charlie: 2400



In [15]:
# 3. Count replacements (3rd argument = count limit)
text = "cat sat on the mat with a cat"
result = re.sub(r"cat", "dog", text, count=1)  # replace only first
print("One replacement:", result)
result_all = re.sub(r"cat", "dog", text)       # replace all
print("All replacements:", result_all)

One replacement: dog sat on the mat with a cat
All replacements: dog sat on the mat with a dog


<a id='s11'></a>
## Section 11: `re.search()` vs `re.match()`

| Feature | `match()` | `search()` |
|---|---|---|
| **Where it looks** | Only at the **start** of the string | **Anywhere** in the string |
| **Use for** | Validating whole strings | Finding something inside text |
| **Returns** | Match object or `None` | Match object or `None` |

> **Rule of thumb:**  
> `match()` = *Does the whole string conform to a format?*  
> `search()` = *Is there something interesting somewhere in this text?*

In [10]:
import re

text = "Welcome back! Your student ID is BT2024 — enjoy the library."
pattern = r"[A-Z]{2}\d{4}"
# match() — only checks position 0
m1 = re.match(pattern, text)
print("match() result:", m1)    # None — pattern not at start

match() result: None


In [11]:
# search() — scans the whole string
m2 = re.search(pattern, text)
if m2:
    print(f"search() found: '{m2.group()}' at position {m2.start()}")

print()

search() found: 'BT2024' at position 33



In [12]:
# Demonstration: match() can still work with ^ and $ for validation
ids = ["BT2024", "CS9999"]
for sid in ids:
    if re.match(r"^[A-Z]{2}\d{4}$", sid):
        print(f"  '{sid}' is a valid student ID")

  'BT2024' is a valid student ID
  'CS9999' is a valid student ID


In [9]:
# search() with flags
paragraph = """Line one is boring.
Line two has the prize: CS2025 here.
Line three."""

m3 = re.search(r"[A-Z]{2}\d{4}", paragraph)
if m3:
    print(f"Found ID: '{m3.group()}' at line {paragraph[:m3.start()].count(chr(10)) + 1}")

Found ID: 'CS2025' at line 2


<a id='s12'></a>
## Section 12: `re.compile()` and Flags

**`re.compile()`** compiles a pattern into a reusable object.  
Useful when the same pattern appears in a loop — avoids recompiling each time.

A compiled pattern object has the same methods: `.match()`, `.search()`, `.findall()`, `.sub()`

**Useful flags:**

| Flag | Short | Effect |
|---|---|---|
| `re.IGNORECASE` | `re.I` | Case-insensitive matching |
| `re.MULTILINE` | `re.M` | `^` and `$` match each line, not just string start/end |
| `re.DOTALL` | `re.S` | `.` matches newlines too |
| `re.VERBOSE` | `re.X` | Allow whitespace and `#` comments in patterns |

In [5]:
import re

# Compile once — use many times
student_id_pattern = re.compile(r"^[A-Z]{2}\d{4}$")

student_ids = ["BT2024", "cs9999", "AB1234", "XYZ123", "CS2025"]
for sid in student_ids:
    if student_id_pattern.match(sid):
        print(f"  PASS: {sid}")
    else:
        print(f"  FAIL: {sid}")

print()

  PASS: BT2024
  FAIL: cs9999
  PASS: AB1234
  FAIL: XYZ123
  PASS: CS2025



In [4]:
# re.IGNORECASE — case-insensitive search
titles = ["SCIENCE FICTION", "Mystery", "science fiction", "Fantasy"]
for title in titles:
    if re.search(r"science fiction", title, re.IGNORECASE):
        print(f"  Sci-fi found: '{title}'")

print()

  Sci-fi found: 'SCIENCE FICTION'
  Sci-fi found: 'science fiction'



In [3]:
# re.VERBOSE — readable multi-line pattern
import re
email_pattern = re.compile(r"""
    [\w.\-]+    # local part: letters, digits, dots, hyphens
    @           # at-sign
    [\w.\-]+    # domain name
    \.          # literal dot
    [a-zA-Z]{2,} # top-level domain: 2+ letters
""", re.VERBOSE)

emails = ["alice@uni.edu", "bad@", "bob@example.co.uk", "ramlal@domai.n"]
for e in emails:
    print(f"  {e!r:25s} valid={bool(email_pattern.fullmatch(e))}")

  'alice@uni.edu'           valid=True
  'bad@'                    valid=False
  'bob@example.co.uk'       valid=True
  'ramlal@domai.n'          valid=False
